# 예제 01. 학과 LLM 서버에 처음 접속하기

전산실험 2 · AI Agent 만들기 · **1주차** · 예상 소요 30분

## 목표
- 학과 서버에 어떤 모델이 올라가 있는지 확인한다
- 첫 응답을 받아낸다
- `messages` 의 세 가지 역할을 이해한다
- 모델이 **이전 대화를 기억하지 못한다**는 사실을 직접 확인한다
- 응답에 토큰이 몇 개 쓰였는지 확인한다

> 셀을 클릭하고 **Shift + Enter** 로 실행합니다.

---
## 0. 준비 — API 키를 안전하게 넣기

수업시간에 공개된 API 키를 **Colab Secrets** 에 저장합니다.

1. 왼쪽 세로 메뉴에서 **열쇠 아이콘(🔑)** 을 클릭
2. **새 secret 추가** — 두 개를 만듭니다
   - `KMU_BASE_URL` = 수업시간에 공개하는 **서버 주소**
   - `KMU_API_KEY` = 수업시간에 공개하는 **API 키**
3. 각각 이 노트북의 **액세스 토글을 켜기** ← 이걸 안 켜면 못 읽습니다

> 서버 주소도 키와 똑같이 관리합니다. 이 저장소는 공개되어 있으므로
> 주소를 코드에 적으면 외부에서 학과 서버를 두드릴 수 있게 됩니다.

키를 코드 셀에 직접 적으면 안 되는 이유는 2주차에 자세히 다룹니다.

In [ ]:
# Colab에 openai 패키지 설치 (이미 있으면 그냥 넘어갑니다)
!pip install -q openai
print("설치 완료")

In [ ]:
# 서버 주소와 API 키 가져오기
#   1순위: Colab Secrets (왼쪽 열쇠 아이콘)
#   2순위: 환경변수
#   3순위: 직접 입력
#
# 주의: 아래에 주소나 키를 직접 타이핑하지 마세요.
#       이 저장소는 공개되어 있어, 적는 순간 전 세계에 노출됩니다.

import os

def get_secret(name, secret=True):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            print(f"{name}: Colab Secrets 에서 가져왔습니다.")
            return v.strip()
    except Exception:
        pass
    if os.environ.get(name):
        print(f"{name}: 환경변수에서 가져왔습니다.")
        return os.environ[name].strip()

    print(f"[{name}] 를 Secrets 에서 찾지 못했습니다. 직접 입력해 주세요.")
    if secret:
        import getpass
        v = getpass.getpass(f"{name}: ").strip()
    else:
        v = input(f"{name}: ").strip()
    os.environ[name] = v
    return v


def get_base_url():
    # 주소를 통째로(.../chat/completions) 저장했어도 알아서 정리합니다
    url = get_secret("KMU_BASE_URL", secret=False).rstrip("/")
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]
    return url


BASE_URL = get_base_url()
API_KEY  = get_secret("KMU_API_KEY")
MODEL    = "gemma4"

print("주소 길이:", len(BASE_URL), "글자  |  키 길이:", len(API_KEY), "글자")
print("(값 자체는 출력하지 않습니다)")

---
## 1. 서버 진단 — 질문하기 전에 먼저 확인한다

바로 질문을 던지지 않고, **가장 단순한 요청**부터 보냅니다.

`GET /v1/models` 는 "이 서버에 어떤 모델이 있나요?" 라고 묻는 요청입니다.
응답이 오면 주소와 키가 모두 정상이라는 뜻입니다.

처음 쓰는 API에서 오류가 났을 때, 주소가 틀린 건지 키가 틀린 건지 모델 이름이
틀린 건지 구분하지 못하면 시간을 크게 낭비합니다.
**되는 것부터 확인하고 하나씩 올라가는 것**이 요령입니다.

In [ ]:
import requests

# BASE_URL, API_KEY, MODEL 은 위 셀에서 Secrets 로부터 읽었습니다.
response = requests.get(
    f"{BASE_URL}/models",
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=15,
)

print("HTTP 상태코드:", response.status_code)
print(response.text[:500])

In [ ]:
# 응답을 보기 좋게 정리
if response.status_code == 200:
    models = [m["id"] for m in response.json()["data"]]
    print(f"사용 가능한 모델 {len(models)}개")
    for name in models:
        mark = "   <-- 우리가 쓸 모델" if name == MODEL else ""
        print("  -", name, mark)

    if MODEL not in models:
        print()
        print("[주의] MODEL 값이 목록에 없습니다.")
        print("       위 목록에서 정확한 이름을 골라 MODEL 을 고쳐 주세요.")
else:
    print("실패했습니다. 아래 셀의 안내를 읽어 보세요.")

### 연결이 안 될 때

| 증상 | 원인 | 해볼 것 |
|---|---|---|
| 연결 자체가 안 됨 | 주소·포트·프로토콜 | 아래 셀로 후보를 자동 탐색 |
| `401` / `403` | 키 문제 | Secrets 이름이 `KMU_API_KEY` 인지, 액세스 토글이 켜졌는지 |
| `404` | 경로 문제 | `KMU_BASE_URL` 끝이 `/v1` 인지 확인 |

아래 셀에서 주소 후보를 한꺼번에 시험해 볼 수 있습니다.

In [ ]:
# 주소 후보 자동 탐색 (연결이 안 될 때만 실행하세요)
# Secrets 에 넣은 주소를 조금씩 바꿔 가며 시험합니다.
from urllib.parse import urlparse

host = urlparse(BASE_URL).hostname or ""
candidates = [BASE_URL]
for scheme in ("https", "http"):
    for port in ("", ":11434", ":8000"):
        candidates.append(f"{scheme}://{host}{port}/v1")

for url in dict.fromkeys(candidates):            # 중복 제거, 순서 유지
    표시 = url.replace(host, host[:4] + "***")    # 전체 주소를 화면에 남기지 않는다
    try:
        r = requests.get(f"{url}/models",
                         headers={"Authorization": f"Bearer {API_KEY}"},
                         timeout=5)
        print(f"{r.status_code}  {표시}")
    except Exception as exc:
        print(f"실패  {표시}  ({type(exc).__name__})")

---
## 2. 첫 호출

이제 실제로 질문을 던집니다.

서버가 **OpenAI 호환** 규격이므로, OpenAI에서 만든 파이썬 라이브러리를
그대로 쓸 수 있습니다. 바꾸는 것은 `base_url` 하나뿐입니다.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=120)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "표준편차를 한 문장으로 설명해 주세요."}
    ],
)

print(response.choices[0].message.content)

### 방금 무슨 일이 일어났나

```
우리     →  {"model": ..., "messages": [...]}   →  서버
서버     →  {"choices": [{"message": {...}}], "usage": {...}}  →  우리
```

`response.choices[0].message.content` 라는 긴 경로를 매번 쓰기는 번거롭습니다.
그래서 함수로 감싸 둡니다. **다음 주에 이 함수들을 모아 파일 하나로 만들 것입니다.**

In [ ]:
def ask(prompt, system=None, **kw):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=MODEL, messages=messages, **kw
    )
    return response.choices[0].message.content.strip()


print(ask("t검정을 한 문장으로 설명해 주세요."))

---
## 3. `messages` 의 세 가지 역할

| 역할 | 뜻 | 언제 쓰나 |
|---|---|---|
| `system` | 모델의 성격·규칙 | 대화 전체에 적용할 지시 |
| `user` | 사용자의 말 | 질문, 요청 |
| `assistant` | 모델이 **앞서 한 말** | 대화를 이어 붙일 때 |

`system` 을 바꾸면 같은 질문에도 답이 달라집니다.

In [ ]:
질문 = "p값이 0.03이면 어떻게 해석하나요?"

for 역할설명 in [
    "너는 통계학과 대학원 조교다. 전문 용어를 그대로 쓰고 두 문장으로 답한다.",
    "너는 통계를 처음 배우는 학부생에게 설명하는 사람이다. 비유를 써서 쉽게 답한다.",
]:
    print("=" * 60)
    print("[system]", 역할설명)
    print("-" * 60)
    print(ask(질문, system=역할설명))
    print()

---
## 4. 모델은 이전 대화를 기억하지 못한다

이것이 오늘 가장 중요한 내용입니다.

서버는 **우리가 보낸 메시지 목록만** 보고 답합니다.
대화가 이어지는 것처럼 보이는 이유는, 우리가 지난 대화를 매번 다시 보내기 때문입니다.

직접 확인해 봅시다.

In [ ]:
# (1) 이전 대화를 보내지 않으면 -- 모델은 아무것도 모릅니다
답1 = ask("제 이름은 김통계입니다. 기억해 주세요.")
print("첫 번째 답:", 답1[:100])
print()

답2 = ask("제 이름이 뭐라고 했죠?")     # 새 요청 = 백지 상태
print("두 번째 답:", 답2[:150])

In [ ]:
# (2) 이전 대화를 함께 보내면 -- 기억하는 것처럼 보입니다
messages = [
    {"role": "user",      "content": "제 이름은 김통계입니다. 기억해 주세요."},
    {"role": "assistant", "content": 답1},          # 모델이 앞서 한 말
    {"role": "user",      "content": "제 이름이 뭐라고 했죠?"},
]

response = client.chat.completions.create(model=MODEL, messages=messages)
print(response.choices[0].message.content)

> **핵심**
> 기억은 모델이 아니라 **우리 코드**가 만드는 것입니다.
> 대화가 길어지면 매번 보내는 양이 늘어나고, 결국 한계에 부딪힙니다.
> 이 문제를 **4주차(메모리)** 에서 본격적으로 다룹니다.

---
## 5. 토큰 — 얼마나 썼는지 확인하기

모델은 글자가 아니라 **토큰** 단위로 텍스트를 처리합니다.
응답 객체의 `usage` 를 보면 이번 호출에 몇 개를 썼는지 알 수 있습니다.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "중심극한정리를 세 문장으로 설명해 주세요."}],
)

u = response.usage
print("입력 토큰 :", u.prompt_tokens)
print("출력 토큰 :", u.completion_tokens)
print("합계      :", u.total_tokens)
print()
print(response.choices[0].message.content)

In [ ]:
# 한국어와 영어는 토큰 소비량이 다릅니다
문장들 = {
    "한국어": "표본의 크기가 커지면 표본평균의 분포는 정규분포에 가까워집니다.",
    "영어":   "As the sample size increases, the distribution of the sample mean approaches a normal distribution.",
}

for 언어, 문장 in 문장들.items():
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": 문장}],
        max_tokens=1,          # 답은 필요 없고 입력 토큰만 세고 싶으므로
    )
    print(f"{언어}: {r.usage.prompt_tokens:3d} 토큰   ({len(문장)} 글자)")

---
## 연습문제

제출하지 않습니다. 직접 해 보고 아래 해설과 비교하세요.

**문제 1.** `system` 메시지를 세 가지로 바꾸어 같은 질문을 던지고,
답의 **어조**와 **길이(글자 수)** 가 어떻게 달라지는지 표로 정리하시오.

**문제 2.** 위 토큰 비교에서 한국어가 영어보다 토큰을 더 많이 쓰는 이유는 무엇일까?
(힌트: 모델이 학습한 데이터의 언어 구성)

**문제 3.** `ask()` 함수에 `max_tokens=30` 을 넣어 호출해 보시오.
응답이 어떻게 되는가? 이런 일이 에이전트에서 일어나면 어떤 문제가 생기겠는가?

In [ ]:
# 여기에 직접 작성해 보세요

<details>
<summary><b>해설 보기</b></summary>

**문제 1.**
`system` 은 대화 전체에 적용되는 지시입니다. 어조뿐 아니라 답의 길이, 용어 수준,
형식(목록/문장)까지 크게 좌우합니다. 에이전트를 만들 때 `system` 은
"이 부품이 무슨 일을 하는가"를 정의하는 자리가 됩니다. 3주차에서 본격적으로 설계합니다.

**문제 2.**
토크나이저는 학습 데이터에서 자주 나온 문자열을 하나의 토큰으로 묶습니다.
대부분의 모델은 영어 텍스트를 훨씬 많이 학습했기 때문에, 영어 단어는 통째로 1토큰이
되는 경우가 많은 반면 한국어는 글자 단위로 잘게 쪼개집니다.
→ **같은 내용이라도 한국어 프롬프트가 컨텍스트를 더 많이 잡아먹습니다.**

**문제 3.**
문장이 중간에서 잘립니다. 에이전트에서는 이것이 치명적입니다.
모델의 출력을 코드로 파싱해야 하는데(3주차), JSON이 중간에 끊기면 파싱이 실패합니다.
`max_tokens` 는 "안전장치"이지 "길이 조절 손잡이"가 아닙니다.
길이를 줄이고 싶으면 프롬프트로 지시하십시오.

</details>

---
## 정리

- 새 API는 **가장 단순한 요청부터** 보내서 진단한다
- `messages` 는 목록이고, 역할은 `system` / `user` / `assistant`
- **모델은 기억하지 못한다.** 기억은 우리 코드가 만든다
- 토큰을 항상 의식한다. 한국어는 더 비싸다

**다음 노트북** → [`02_chat_params.ipynb`](https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week01/02_chat_params.ipynb)
temperature가 응답을 얼마나 흔드는지 직접 측정합니다.